<a href="https://colab.research.google.com/github/gnutan181/Training-model/blob/main/FineTuningofBERTUsingLoRaAndQLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import torch
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
      Trainer,
    TrainingArguments,BitsAndBytesConfig,
    pipeline)
from sklearn.metrics import accuracy_score, f1_score

from peft import LoraConfig, TaskType, get_peft_model,prepare_model_for_kbit_training

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Load the dataset
# ─────────────────────────────────────────────────────────────────────────────

dataset = load_dataset('dair-ai/emotion')

df = dataset['train'].to_pandas()

label_names = dataset['train'].features['label'].names
df['label_text'] = df['label'].apply(lambda x : label_names[x])



In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Tokenization
# ─────────────────────────────────────────────────────────────────────────────
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Train / Test/ Validation Split (70/20/10)
# ─────────────────────────────────────────────────────────────────────────────
#70% of data = Train data, 20% of data = Test Data, 10% of data = Validation data
train, test = train_test_split(df, test_size=0.3, stratify=df['label_text'])
test, validation = train_test_split(test, test_size=1/3, stratify=test['label'])



In [5]:


# ─────────────────────────────────────────────────────────────────────────────
# 4.Building a hugging face DatasetDict
# ─────────────────────────────────────────────────────────────────────────────

dataset = DatasetDict({
    'train' : Dataset.from_pandas(train,preserve_index=False),
    'test' : Dataset.from_pandas(test,preserve_index=False),
    'validation' : Dataset.from_pandas(validation,preserve_index=False)
})



In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# 5.Tokenize all Splits via dataset.map
# ─────────────────────────────────────────────────────────────────────────────

def tokenize(batch):
    return tokenizer(batch['text'],padding =True,truncation=True)


encoded = dataset.map(tokenize,batched=True,batch_size =None)


Map:   0%|          | 0/11200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# 6.Building the Model
# ─────────────────────────────────────────────────────────────────────────────
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}


In [8]:
%pip install -U transformers accelerate peft "bitsandbytes>=0.46.1"

In [9]:
# !pip install -U bitsandbytes>=0.46.1

import sys
import site
# Force refresh package paths so the newly installed bitsandbytes is available
importlib_reload = False
try:
    import bitsandbytes
except ImportError:
    site.main()
    importlib_reload = True

# ─────────────────────────────────────────────────────────────────────────────
# 6a. QLoRA Configuration
# ─────────────────────────────────────────────────────────────────────────────
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

if torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

if device.type == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["classifier", "pooler"]
    )
else:
    bnb_config = None

config = AutoConfig.from_pretrained(model_checkpoint, label2id=label2id, id2label=id2label)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    config=config,
    quantization_config=bnb_config,
    device_map="auto" if device.type == "cuda" else None
)

if bnb_config is not None:
    model = prepare_model_for_kbit_training(model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
!pip install -U torchao

# ─────────────────────────────────────────────────────────────────────────────
# 6a. LoRA Configuration
# ─────────────────────────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n================== BEFORE PEFT ===================")
print(f"Total Params: {total_params}")
print(f"Total Trainable Params: {trainable_params}")

peft_config = LoraConfig(
    task_type= TaskType.SEQ_CLS,
    r=16,
    lora_alpha= 32,
    lora_dropout=0.05,
    target_modules=["query", "key", "value"],
    bias="none"
)

model = get_peft_model(model, peft_config)
print("\n================== AFTER PEFT ===================")
model.print_trainable_parameters()

model = model.to(device)



================== BEFORE PEFT ===================
Total Params: 66724614
Total Trainable Params: 0

================== AFTER PEFT ===================
trainable params: 889,350 || all params: 110,376,204 || trainable%: 0.8057


In [11]:

# ─────────────────────────────────────────────────────────────────────────────
# 7.Training Args
# ─────────────────────────────────────────────────────────────────────────────
batch_size = 64
training_dir = "bert_base_uncased_trained__lora_model"

training_args = TrainingArguments(output_dir=training_dir,
                                  num_train_epochs= 2,
                                  learning_rate= 2e-5,
                                  per_device_eval_batch_size=batch_size,
                                  per_device_train_batch_size=batch_size,
                                  weight_decay=0.01,
                                  eval_strategy='epoch',
                                  disable_tqdm=False)


In [12]:

# ─────────────────────────────────────────────────────────────────────────────
# 8. Compute Metrics
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "F1 Score": f1}



In [13]:

# ─────────────────────────────────────────────────────────────────────────────
# 9. Trainer
# ─────────────────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset= encoded['train'],
    eval_dataset=encoded['validation'],
    processing_class=tokenizer
)

In [14]:
%pip install --upgrade "accelerate>=1.1.0"

In [15]:

# ─────────────────────────────────────────────────────────────────────────────
# 10. Training
# ─────────────────────────────────────────────────────────────────────────────
trainer.train()



/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 score
1,No log,1.566744,0.351250,0.216666
2,No log,1.561300,0.360625,0.232926


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=350, training_loss=1.5966407993861607, metrics={'train_runtime': 473.7973, 'train_samples_per_second': 47.278, 'train_steps_per_second': 0.739, 'total_flos': 907221956659200.0, 'train_loss': 1.5966407993861607, 'epoch': 2.0})

In [16]:


# ─────────────────────────────────────────────────────────────────────────────
# 11. Evaluate the model
# ─────────────────────────────────────────────────────────────────────────────
trainer.evaluate()


Training Loss,Validation Loss,Epoch,Accuracy,F1 score
No log,1.561300,2,0.360625,0.232926


{'eval_loss': 1.5613003969192505,
 'eval_accuracy': 0.360625,
 'eval_F1 Score': 0.2329257594354591}

In [17]:

# ─────────────────────────────────────────────────────────────────────────────
# 12. Save the Model
# ─────────────────────────────────────────────────────────────────────────────
SAVE_DIR = "bert_base_uncased_text-classification-lora_model"
trainer.save_model(SAVE_DIR)


In [20]:
print(SAVE_DIR)
# ─────────────────────────────────────────────────────────────────────────────
# 12. Inferencing the locally saved Model
# ─────────────────────────────────────────────────────────────────────────────
classifier = pipeline('text-classification', model=SAVE_DIR)

# for r in classifier([
#     'I like ML',
#     'I started annoyed with laptops',
#     'I am low today',
#     'I am tensed if I am not going to get the job in ML',
#     'I am worried with the current job market'
# ]):
#     print(r)

bert_base_uncased_text-classification-lora_model


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert_base_uncased_text-classification-lora_model
Key                                       | Status   |                                                                                       
------------------------------------------+----------+---------------------------------------------------------------------------------------
classifier.modules_to_save.default.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([2])          
classifier.modules_to_save.default.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

[transformers] BertForSequenceClassification LOAD REPORT from: bert_base_uncased_text-classification-lora_model
Key                                       | Status   |                                                                                       
------------------------------------------+----------+---------------------------------------------------------------------------------------
classifier.modules_to_save.default.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([2])          
classifier.modules_to_save.default.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/72 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert_base_uncased_text-classification-lora_model
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The model 'BertForMaskedLM' is not supported for text-classification. Supported models are ['PeftModelForSequenceClassification', 'AlbertForSequenceClassification', 'ArceeForSequenceClassification', 'AXK1ForSequenceClassification', 'AXK2ForSequenceClassification', 'BartForSequenceClassification', 'BertForSequenceClassification', 'BigBirdForSequenceClassification', 'BigBirdPegasusForSequenceClassification', 'BioGptForSequenceClassification', 'BloomForSequenceClassification', 'CamembertForSequenceClassification', 'CanineForSequenceClassification', 'CohereCompassTextForSequenceClassification', 'ConvBertForSequence

In [27]:
model_dir
label_names

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [25]:

from peft import PeftConfig, PeftModel

# model_dir = your saved folder path
# label_names = your original label names, in training order

# 1. Read the saved adapter configuration.
peft_config = PeftConfig.from_pretrained(model_dir)

# 2. Load BERT with the correct classification head.
base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=len(label_names),
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)

# 3. Restore your trained adapter.
model = PeftModel.from_pretrained(base_model, model_dir)

# 4. Merge the adapter into BERT for the standard pipeline.
model = model.merge_and_unload()
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_dir)

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

print("Model:", type(classifier.model).__name__)
print("Labels:", classifier.model.config.id2label)

texts = [
    "I like ML",
    "I started annoyed with laptops",
    "I am low today",
    "I am tensed if I am not going to get the job in ML",
]

for text, result in zip(texts, classifier(texts, truncation=True)):
    print(f"{text} -> {result}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: BertForSequenceClassification
Labels: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
I like ML -> {'label': 'sadness', 'score': 0.3165039122104645}
I started annoyed with laptops -> {'label': 'sadness', 'score': 0.30531468987464905}
I am low today -> {'label': 'joy', 'score': 0.3645387291908264}
I am tensed if I am not going to get the job in ML -> {'label': 'sadness', 'score': 0.3123951256275177}


In [30]:
print("Completed training steps:", trainer.state.global_step)

print("\nValidation results:")
print(trainer.evaluate())

print("\nRecent training logs:")
for entry in trainer.state.log_history[-10:]:
    print(entry)

Completed training steps: 350

Validation results:


Training Loss,Validation Loss,Epoch,Accuracy,F1 score
No log,1.561300,2,0.360625,0.232926


{'eval_loss': 1.5613003969192505, 'eval_accuracy': 0.360625, 'eval_F1 Score': 0.2329257594354591}

Recent training logs:
{'eval_loss': 1.5667437314987183, 'eval_accuracy': 0.35125, 'eval_F1 Score': 0.21666601563153126, 'eval_runtime': 8.2625, 'eval_samples_per_second': 193.645, 'eval_steps_per_second': 3.026, 'epoch': 1.0, 'step': 175}
{'eval_loss': 1.5613003969192505, 'eval_accuracy': 0.360625, 'eval_F1 Score': 0.2329257594354591, 'eval_runtime': 8.2752, 'eval_samples_per_second': 193.349, 'eval_steps_per_second': 3.021, 'epoch': 2.0, 'step': 350}
{'train_runtime': 473.7973, 'train_samples_per_second': 47.278, 'train_steps_per_second': 0.739, 'total_flos': 907221956659200.0, 'train_loss': 1.5966407993861607, 'epoch': 2.0, 'step': 350}
{'eval_loss': 1.5613003969192505, 'eval_accuracy': 0.360625, 'eval_F1 Score': 0.2329257594354591, 'eval_runtime': 8.2709, 'eval_samples_per_second': 193.45, 'eval_steps_per_second': 3.023, 'epoch': 2.0, 'step': 350}
{'eval_loss': 1.5613003969192505, 'eva